# 00 — Setup & base de données

Ce notebook initialise la base de données PostgreSQL utilisée par cardiolab.
Il couvre trois opérations :

1. **Vérification** — connexion, état des migrations, existence des tables
2. **Init / mise à jour** — application des migrations manquantes (idempotent)
3. **Reset** — suppression totale + recréation (⚠️ irréversible)
4. **Drop only** — suppression des tables sans recréation (tests)

**Prérequis :** fichier `.env` (production) ou `.env_test` (tests) à la racine du dépôt
avec `DB_HOST`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`.
Modifiez `ENV_FILE` dans la première cellule pour basculer entre les deux.

In [1]:
import os
from pathlib import Path

# ── Fichier d'environnement ──────────────────────────────────────────────────
ENV_FILE = Path("../.env_test")   # ← remplacer par Path("../.env") en production
# ────────────────────────────────────────────────────────────────────────────

try:
    from dotenv import load_dotenv
    if ENV_FILE.exists():
        load_dotenv(ENV_FILE)
        print(f"✓  {ENV_FILE.name} chargé depuis {ENV_FILE.resolve()}")
    else:
        print(f"⚠️  Fichier {ENV_FILE.name} introuvable — vérifiez le chemin")
except ImportError:
    print("⚠️  python-dotenv non installé : pip install python-dotenv")

import psycopg2

from cardiolab.database import HRVRepository, run_migrations
from cardiolab.database.migrator import MIGRATIONS_DIR

print("✓  Imports OK")

✓  .env_test chargé depuis /home/jmilon/Projet-Pro-DevSecOps/Programmation/GitLab/cardioanalysis/cardiolab/.env_test
✓  Imports OK


## 1 — Connexion

Test de la connexion PostgreSQL. Si cette cellule échoue, vérifiez :
- que PostgreSQL tourne (`pg_isready`)
- les variables dans `.env`
- que la base de données existe (`createdb cardiolab`)

In [2]:
def _open_conn():
    return psycopg2.connect(
        host=os.environ["DB_HOST"],
        dbname=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        port=int(os.environ.get("DB_PORT", 5432)),
    )

try:
    conn = _open_conn()
    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        version = cur.fetchone()[0]
    conn.close()
    print(f"✓  Connexion OK\n   {version}")
except KeyError as e:
    print(f"✗  Variable d'environnement manquante : {e}")
except psycopg2.OperationalError as e:
    print(f"✗  Connexion impossible : {e}")

✓  Connexion OK
   PostgreSQL 14.13 (Debian 14.13-1.pgdg110+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 10.2.1-6) 10.2.1 20210110, 64-bit


## 2 — État des migrations

cardiolab utilise un système de migrations SQL versionnées (`V001__`, `V002__`, …)
stockées dans `cardiolab/database/migrations/`.

La table `schema_migrations` trace les versions déjà appliquées pour garantir
l'idempotence : rejouer les migrations sur une base à jour est toujours sans effet.

In [3]:
conn = _open_conn()

# Vérifier si schema_migrations existe
with conn.cursor() as cur:
    cur.execute(
        "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
        "WHERE table_schema='public' AND table_name='schema_migrations');"
    )
    tracking_exists = cur.fetchone()[0]

applied = set()
if tracking_exists:
    with conn.cursor() as cur:
        cur.execute("SELECT version, applied_at FROM schema_migrations ORDER BY version;")
        rows = cur.fetchall()
    applied = {r[0] for r in rows}
    print("Migrations appliquées :")
    for version, applied_at in rows:
        print(f"  ✓  {version:<52}  {applied_at:%Y-%m-%d %H:%M}")
else:
    print("  (schema_migrations absente — première initialisation)")

# Migrations en attente
pending = sorted(f for f in MIGRATIONS_DIR.glob("V*.sql") if f.stem not in applied)
if pending:
    print("\nMigrations en attente :")
    for f in pending:
        print(f"  ○  {f.stem}")
else:
    print("\n  ✅  Base de données à jour — aucune migration en attente.")

conn.close()

Migrations appliquées :
  ✓  V001__initial_schema                                  2026-06-09 16:39
  ✓  V002__add_apen_sampen_ortho_metrics                   2026-06-09 16:39
  ✓  V003__add_training_sessions                           2026-06-09 16:39
  ✓  V004__add_user_profiles                               2026-06-09 16:39

  ✅  Base de données à jour — aucune migration en attente.


## 3 — Appliquer les migrations manquantes

Cette cellule est **idempotente** : si la base est déjà à jour, elle ne fait rien.
À relancer après chaque mise à jour de cardiolab pour appliquer les nouvelles migrations.

In [4]:
with HRVRepository.from_env() as repo:
    applied_now = run_migrations(repo._conn)  # noqa: SLF001

if applied_now:
    for v in applied_now:
        print(f"  ✓  appliquée : {v}")
    print(f"\n✅  {len(applied_now)} migration(s) appliquée(s).")
else:
    print("✅  Base de données déjà à jour.")

✅  Base de données déjà à jour.


## 4 — État des tables

Vérification de l'existence de chaque table cardiolab.

In [5]:
_TABLES = [
    ("hrv_features",          "Resting HRV"),
    ("hrv_orthostatic",       "Orthostatic"),
    ("hrv_coherence",         "Cardiac coherence"),
    ("hrv_hrr",               "Heart Rate Recovery"),
    ("hrv_drift",             "Cardiac drift"),
    ("hrv_vo2max",            "VO2max estimation"),
    ("hrv_raw_sessions",      "Raw RR intervals"),
    ("hrv_training_sessions", "Training sessions"),
    ("user_profiles",         "User profiles"),
    ("schema_migrations",     "Migration tracking"),
]

conn = _open_conn()
print(f"{'Table':<28} {'Description':<22} État")
print("-" * 60)
with conn.cursor() as cur:
    for table, desc in _TABLES:
        cur.execute(
            "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
            "WHERE table_schema='public' AND table_name=%s);",
            (table,),
        )
        exists = cur.fetchone()[0]
        status = "✓  existe" if exists else "○  absente"
        print(f"{table:<28} {desc:<22} {status}")
conn.close()

Table                        Description            État
------------------------------------------------------------
hrv_features                 Resting HRV            ✓  existe
hrv_orthostatic              Orthostatic            ✓  existe
hrv_coherence                Cardiac coherence      ✓  existe
hrv_hrr                      Heart Rate Recovery    ✓  existe
hrv_drift                    Cardiac drift          ✓  existe
hrv_vo2max                   VO2max estimation      ✓  existe
hrv_raw_sessions             Raw RR intervals       ✓  existe
hrv_training_sessions        Training sessions      ✓  existe
user_profiles                User profiles          ✓  existe
schema_migrations            Migration tracking     ✓  existe


---

## ⚠️ RESET — Suppression totale et recréation

> **Irréversible.** Toutes les données enregistrées seront perdues.
> Utile uniquement pour repartir d'une base propre (dev, tests).

Pour activer, passez `CONFIRM_RESET = True` dans la cellule suivante.

In [6]:
CONFIRM_RESET = True  # ← passer à True pour effectuer le reset

if not CONFIRM_RESET:
    print("Reset désactivé (CONFIRM_RESET = False).")
else:
    from psycopg2 import sql as pg_sql

    print("Suppression des tables...")
    conn = _open_conn()
    with conn.cursor() as cur:
        for table, _ in reversed(_TABLES):
            cur.execute(
                pg_sql.SQL("DROP TABLE IF EXISTS {} CASCADE;").format(
                    pg_sql.Identifier(table)
                )
            )
            print(f"  ✗  {table}")
    conn.commit()
    conn.close()

    print("\nRéapplication des migrations...")
    with HRVRepository.from_env() as repo:
        applied_now = run_migrations(repo._conn)  # noqa: SLF001
    for v in applied_now:
        print(f"  ✓  {v}")
    print(f"\n✅  Reset terminé — {len(applied_now)} migration(s) appliquée(s).")

Suppression des tables...
  ✗  schema_migrations
  ✗  user_profiles
  ✗  hrv_training_sessions
  ✗  hrv_raw_sessions
  ✗  hrv_vo2max
  ✗  hrv_drift
  ✗  hrv_hrr
  ✗  hrv_coherence
  ✗  hrv_orthostatic
  ✗  hrv_features

Réapplication des migrations...
  ✓  V001__initial_schema
  ✓  V002__add_apen_sampen_ortho_metrics
  ✓  V003__add_training_sessions
  ✓  V004__add_user_profiles

✅  Reset terminé — 4 migration(s) appliquée(s).


---

## 5 — Drop all tables

Supprime toutes les tables cardiolab **sans les recréer**.
Utile pour nettoyer une base de test après une session.

> **Différence avec le Reset (§3) :** le reset supprime ET recrée via les migrations.
> Ce drop est définitif — relancez les cellules 3 et 4 pour réinitialiser.

In [7]:
def drop_all_tables():
    """Supprime toutes les tables cardiolab (sans recréation)."""
    from psycopg2 import sql as pg_sql
    conn = _open_conn()
    with conn.cursor() as cur:
        for table, _ in reversed(_TABLES):
            cur.execute(
                pg_sql.SQL("DROP TABLE IF EXISTS {} CASCADE;").format(
                    pg_sql.Identifier(table)
                )
            )
            print(f"  ✗  {table}")
    conn.commit()
    conn.close()
    print("\n✅  Toutes les tables supprimées.")


CONFIRM_DROP = False  # ← passer à True pour effectuer le drop

if CONFIRM_DROP:
    drop_all_tables()
else:
    print("Drop désactivé (CONFIRM_DROP = False).")
    print("Passez CONFIRM_DROP = True pour supprimer toutes les tables.")

Drop désactivé (CONFIRM_DROP = False).
Passez CONFIRM_DROP = True pour supprimer toutes les tables.
